## **Echantillonage probabiliste avec Python: Implémentation du sondage aléotoire simple, de la stratification et du sondage à probabilités inégales**


### **Fonctions necessaires à l'implémentation**

In [896]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
import warnings

warnings.filterwarnings('ignore')


def equi_inclusion(df,u):
    """
    Cette fonction permet de recuper l'indexe du 
    plus petit élement d'une serie iméditement supérieur à une valuer donnée.
    """
    db=df-u
    index=db[db==db[db>=0].min()].index
    return index[0]


def select_sys(base_sondage,
               taille,
               poids=None,
               random_state=42
              ):
    """
    fonction pour selectionnner un échantillon probabiliste suivant la 
    méthode de tirage systématique.
    Il est demandé de préciser outre la base de sondage (DataFrame) et 
    les colonnes identifiant et nom des unités, le pas qui est le random state 
    et la taille de l'échantillon. 
    """
    import pandas as pd
    import numpy as np
    import math

    # génération du nombre aléatoire
    nb_alea=np.random.default_rng(seed=random_state).random()


    # vérifier si le paramètre poids est None. S'il est None c'est un sondage à probabilité égale
    
    # 1er cas: probabilité égale:
    
    # création des numéros d'ordres
    base_sondage["id_sys"]=range(1,len(base_sondage)+1)
    
    if poids==None:
        if taille>= len(base_sondage):
            print("la taille de l'échantillon est supérieur ou égal à la taille de la base de sondage")
        else:
            k=len(base_sondage)/taille if taille!=0 else 0
            select=[]
            for i in range(taille):
                select.append(math.floor(k*nb_alea+(i*k))+1)
            df = base_sondage[base_sondage["id_sys"].isin(select)]
            return select,df

    # 2er cas: probabilité inégale:
    else:
        c_poids=base_sondage[poids].apply(pd.to_numeric,errors="coerce").sum()
        for idx,row in base_sondage.iterrows():
            base_sondage.loc[idx,"poids_relatif"]=base_sondage.loc[idx,poids].sum()/c_poids
            base_sondage.loc[idx,"p_inclusion"]=base_sondage.loc[idx,"poids_relatif"]*taille
            base_sondage.loc[idx,"cum_p_inclusion"]=base_sondage.loc[0:idx,"p_inclusion"].sum()
        
        p_inclusion=base_sondage["p_inclusion"]>1
        # vérifier que les probabilié d'inclusion sont toutes inférieur à <1
        if p_inclusion.sum()==0:
            select=[nb_alea+i for i in range(taille)]
            select_cor=[equi_inclusion(base_sondage["cum_p_inclusion"],u) for u in select]
            df = base_sondage[base_sondage.index.isin(select_cor)]
            return select_cor,df
        else:
            select_cor_p=[next(u for u in p_inclusion if u==False),None]
            if len(select_cor_p)==taille:
                df=base_sondage[base_sondage["cum_p_inclusion"].isin(select_cor_p)]
                return select_cor_p,df
            else:
                long_reste=taille-len(select_cor_p)
                select=[nb_alea+i for i in range(long_reste)]
                select_cor=[equi_inclusion(base_sondage["cum_p_inclusion"],u) for u in select]
                df = base_sondage[base_sondage.index.isin(select_cor+select_cor_p)]
                select=select_cor+select_cor_p
                return select, df
## prise en compte de la stratification dans l'échantillonage

def select_sys_complete(base_sondage, 
                        taille, 
                        strate,
                        poids_u=None, # précise le type de sondage. si poids_u précisé alors sondage à probabilité inégale
                        poids=None,
                        random_state=42, 
                        share=0
                       ):
    """
    cette fonction permet de prendre en compte la stratification dans le plan de sondage
    les paramettres sont 
    """
    stra_stat=base_sondage[strate].value_counts()
    strates=stra_stat.index
    df=pd.DataFrame() # Dataframe devant contenir les échantillons par strate
    nbre_strate=len(strates)
    dic_strates={}
    poids_strate={}
    select=[] # Numéro des unités tirées 
    
    #calcul de la taille de l'échantillon par strate
    
    if share not in (0,1):
        print("L'option share doit être 0 ou 1")
        return select,df
    elif  share==0:
        for stra in strates:
            # échantillonnage pour la grappe 
            dic_strates[stra]=int(round(taille*stra_stat[stra]/stra_stat.sum()))
        total_poids=pd.Series(dic_strates.values()).sum()
        if total_poids<taille:
            print(f"Les strates sont nombreuses et ne permettent pas de repartir assez bien l'echantillon de taille {taille}")
            return select,df
        else:
            for stra in strates:
                select_stra,df_ech_stra=select_sys(base_sondage[base_sondage[strate]==stra],
                                                   dic_strates[stra],
                                                   poids_u,
                                                   random_state
                                                  )
                df=pd.concat([df,df_ech_stra])
                select.extend(select_stra)
            return select,df
    elif share==1 and poids!=None:
        for stra in strates:
            dic_strates[stra]=stra_stat[stra]*base_sondage[base_sondage[strate]==stra][poids].sum()
        total_strates=pd.Series(dic_strates.values()).sum()
        for stra in strates:
            poids_strate[stra]=int(round(taille*dic_strates[stra]/total_strates))
        total_poids=pd.Series(poids_strate.values()).sum()
        if total_poids<0:
            print(f"Les strates sont nombreuses et ne permettent pas de repartir assez bien l'echantillon de taille {taille}")
            return select,df
        else:
            for stra in strates:
                select_stra,df_ech_stra=select_sys(base_sondage[base_sondage[strate]==stra],
                                               poids_strate[stra],
                                               poids_u,
                                               random_state
                                              )
                df=pd.concat([df,df_ech_stra])
                select.extend(select_stra)
            return select,df
    else:
        print("share=1 nécessite une variable de poids")
        return select,df

### **Test de l'implementation sur l'échantillonnage des ZD pour une étude dans la ville de Bafia au Cameroun**

In [884]:
## importation de la base de sondage
base_sondage=pd.read_excel("./exemple/zd_bafia.xlsx")

In [886]:
# visualisation des données
base_sondage.head()

,Region,Departemen,Arrondisse,Canton,Chefferie,Bloc_Loc,Ville,Milieu,ZD_4RGPH,ZD,ZE,ZS,ZC,Ménages_E,POP_E,Ménages_R,POP_R,N_A_R,Titre,ZD3
0,CENTRE,MBAM ET INOUBOU,BAFIA,BAFIA,BIABEREBE,BIABEREBE,NaN,1,33,4,AD,NaN,NaN,179,895,0,0,0,Arrondissement de BAFIA : ZD N° 5 ZE ( AD ),1
1,CENTRE,MBAM ET INOUBOU,BAFIA,BAFIA,BIABEREBE,BIABEREBE,NaN,1,40,16,AD,NaN,NaN,266,1170,0,0,0,Arrondissement de BAFIA : ZD N° 17 ZE ( AD ),1
2,CENTRE,MBAM ET INOUBOU,BAFIA,BAFIA,BIABEREBE,BIABEREBE,NaN,1,21,6,AB,NaN,NaN,237,1055,0,0,0,Arrondissement de BAFIA : ZD N° 7 ZE ( AB ),1
3,CENTRE,MBAM ET INOUBOU,BAFIA,BAFIA,BIABEREBE,BIABEREBE,NaN,1,51,7,AB,NaN,NaN,225,1277,0,0,0,Arrondissement de BAFIA : ZD N° 8 ZE ( AB ),1
4,CENTRE,MBAM ET INOUBOU,BAFIA,BAFIA,BIABEREBE,BIABEREBE,NaN,1,23,23,AL,NaN,NaN,167,965,0,0,0,Arrondissement de BAFIA : ZD N° 24 ZE ( AL ),1


### Probabilité égale

In [888]:
select,df=select_sys(base_sondage[["Canton","Chefferie","ZD","Ménages_E","POP_E"]],random_state=0,taille=5)
df

,Canton,Chefferie,ZD,Ménages_E,POP_E,id_sys
7,BAFIA,BIABETOM,34,171,1047,8
18,BAFIA,BIABEYAKAN,19,205,991,19
30,BAFIA,LABLE,45,206,1140,31
41,BAFIA,RIMIS I,27,165,877,42
52,BAFIA,RIMIS II,30,261,1182,53


### Probabilité inégale

In [862]:
select,df=select_sys(base_sondage[["Canton","Chefferie","ZD","Ménages_E","POP_E"]],random_state=0,poids="POP_E",taille=5)
df

,Canton,Chefferie,ZD,Ménages_E,POP_E,id_sys,poids_relatif,p_inclusion,cum_p_inclusion
7,BAFIA,BIABETOM,34,171,1047,8,0.017706,0.088529,0.692676
19,BAFIA,BIABEYAKAN,17,232,992,20,0.016776,0.083879,1.700912
30,BAFIA,LABLE,45,206,1140,31,0.019279,0.096393,2.689361
42,BAFIA,RIMIS I,26,196,913,43,0.015440,0.077199,3.704615
52,BAFIA,RIMIS II,30,261,1182,53,0.019989,0.099944,4.656875


### Stratification proportionnelle (probabilité inégale)

In [900]:
select,df=select_sys_complete(base_sondage[["Canton","Chefferie","Bloc_Loc","ZD","Ménages_E","POP_E"]],
                              random_state=0,
                              poids_u="POP_E",
                              poids="POP_E",
                              taille=5,
                              strate="Chefferie",
                              share=0
                             )
df

,Canton,Chefferie,Bloc_Loc,ZD,Ménages_E,POP_E,id_sys,poids_relatif,p_inclusion,cum_p_inclusion
44,BAFIA,RIMIS I,CHEFFERIE,29,258,1250,6,0.133647,0.133647,0.668235
52,BAFIA,RIMIS II,CHEFFERIE,30,261,1182,5,0.134029,0.134029,0.642023
4,BAFIA,BIABEREBE,BIABEREBE,23,167,965,5,0.135059,0.135059,0.750455
11,BAFIA,BIABETOM,BIABETOM,31,203,991,5,0.150516,0.150516,0.737849
17,BAFIA,BIABEYAKAN,BIABEYAKAN,20,195,905,4,0.141694,0.141694,0.689526


### Stratification optimale (probabilité égale)

In [907]:
select,df=select_sys_complete(base_sondage[["Canton","Chefferie","Bloc_Loc","ZD","Ménages_E","POP_E"]],
                              random_state=0,
                              poids="POP_E",
                              taille=5,
                              strate="Chefferie",
                              share=1
                             )
df

,Canton,Chefferie,Bloc_Loc,ZD,Ménages_E,POP_E,id_sys
44,BAFIA,RIMIS I,CHEFFERIE,29,258,1250,6
53,BAFIA,RIMIS II,CHEFFERIE,10,130,884,6
4,BAFIA,BIABEREBE,BIABEREBE,23,167,965,5
11,BAFIA,BIABETOM,BIABETOM,31,203,991,5
17,BAFIA,BIABEYAKAN,BIABEYAKAN,20,195,905,4
